# Pipeline IR & IE Bước 4 - Fact-aware Evidence Reranking & Đánh giá NLI End-to-End
## Module 9 & Pipeline Validation: Reranking → NLI Inputs → BamiBERT Inference

---

### Điểm quan trọng về Kiến trúc Đồ án:
1. **IR & IE không train NLI**: IR/IE chịu trách nhiệm sàng lọc ra **Evidence mới** có độ tin cậy và sự thật cao nhất.
2. **Tạo dữ liệu đầu vào chuẩn hóa NLI**: `(Claim, Evidence, Label)` xuất thành file CSV.
3. **Model-specific Preprocessing & Inference**:
   - Chạy mô hình NLI BamiBERT đã huấn luyện (`Exp006_PrefixPrompt_SOTA`) trên 3 thí nghiệm đối chứng:
     - **Thí nghiệm A (Gold Evidence):** Claim + Bằng chứng chuẩn (Upper-bound lý tưởng).
     - **Thí nghiệm B (BM25 Evidence):** Claim + Bằng chứng do BM25 Top 1 tìm thấy.
     - **Thí nghiệm C (BM25 + IE Reranking):** Claim + Bằng chứng do Fact-aware Reranker chọn lọc.
   - Chứng minh định lượng mức độ đóng góp của Fact-aware Reranking đối với bài toán Fact-checking!


In [1]:
# ======================================================================
# 1. KHAI BÁO CÁC THƯ VIỆN CẦN THIẾT
# ======================================================================
import json  # Đọc và ghi file định dạng JSON
import os  # Tương tác với hệ điều hành
import sys  # Lấy thông tin môi trường thực thi
from pathlib import Path  # Xử lý đường dẫn file/thư mục

import matplotlib.pyplot as plt  # Vẽ đồ thị trực quan hóa
import numpy as np  # Tính toán ma trận và mảng số học
import pandas as pd  # Xử lý bảng dữ liệu DataFrame
import torch  # Thư viện PyTorch phục vụ chạy mô hình Deep Learning
import torch.nn as nn  # Các lớp mạng nơ-ron PyTorch
import torch.nn.functional as F  # Các hàm kích hoạt và hàm mất mát
from torch.utils.data import TensorDataset, DataLoader  # Quản lý batch dữ liệu
from sklearn.metrics import accuracy_score, f1_score  # Thước đo độ chính xác và Macro F1

# ======================================================================
# 2. TỰ ĐỘNG ĐỊNH VỊ THƯ MỤC GỐC VÀ ĐỌC BẢNG ĐẶC TRƯNG FACT (TỪ BƯỚC 3)
# ======================================================================
current_dir = Path.cwd().resolve()
candidates = [current_dir, current_dir.parent, current_dir.parent.parent]
PROJECT_ROOT = current_dir
for cand in candidates:
    if (cand / 'data').exists() and (cand / 'notebooks').exists():
        PROJECT_ROOT = cand
        break

OUTPUT_DIR = PROJECT_ROOT / 'data/processed/retrieval'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# File đầu vào: Bảng Top-5 câu kèm điểm Fact đã tính ở Bước 3
INPUT_PATH = OUTPUT_DIR / 'evidence_features.csv'
df_feats = pd.read_csv(INPUT_PATH)

# ======================================================================
# 3. CHUẨN HÓA ĐIỂM BM25 VỀ ĐOẠN [0, 1] CHO TỪNG BÀI BÁO
# ======================================================================
# Tìm điểm BM25 cao nhất trong 5 câu của từng câu tuyên bố
df_feats['bm25_max'] = df_feats.groupby('claim_id')['bm25_score'].transform('max')
# Chia cho giá trị max để điểm BM25 nằm gọn trong khoảng [0, 1] cân bằng với Fact Score
df_feats['bm25_norm'] = df_feats['bm25_score'] / (df_feats['bm25_max'] + 1e-6)

# ======================================================================
# 4. TÍNH ĐIỂM KẾT HỢP ĐA TIÊU CHÍ VÀ XẾP HẠNG LẠI (RERANKING)
# ======================================================================
# Tỷ lệ vàng đã tối ưu qua Grid Search: 70% Ngữ cảnh từ vựng (BM25) + 30% Sự thật (Fact)
ALPHA = 0.70
BETA = 0.30
# Công thức tính điểm cuối cùng: Điểm càng cao thì câu càng đúng cả về từ vựng lẫn số liệu/tên riêng
df_feats['rerank_score'] = round(ALPHA * df_feats['bm25_norm'] + BETA * df_feats['fact_score'], 4)

# Lưu lại thứ hạng cũ của BM25
df_feats['bm25_rank'] = df_feats['rank']
# Sắp xếp lại 5 câu theo điểm mới từ cao xuống thấp và đánh số thứ tự xếp hạng mới (1 đến 5)
df_feats['rerank_rank'] = df_feats.sort_values(['claim_id', 'rerank_score'], ascending=[True, False]).groupby('claim_id').cumcount() + 1

# Lưu kết quả xếp hạng lại xuống file CSV
output_rerank_path = OUTPUT_DIR / 'reranked_evidence.csv'
df_feats.to_csv(output_rerank_path, index=False)
print(f'✓ Đã lưu bảng xếp hạng lại tại: {output_rerank_path.name}')


✓ Đã lưu bảng xếp hạng lại tại: reranked_evidence.csv


In [2]:
# ======================================================================
# ĐÁNH GIÁ ĐỘ PHỦ (RECALL@K) VÀ THỨ HẠNG (MRR) TRƯỚC VÀ SAU RERANKING
# ======================================================================
# Chỉ lọc tập các câu SUPPORTED và REFUTED (nhãn != 2) để kiểm tra vì chỉ chúng mới có Bằng chứng Vàng
eval_df = df_feats[df_feats['label'] != 2].copy()
n_claims = eval_df['claim_id'].nunique()  # Tổng số lượng câu tuyên bố cần kiểm tra

def eval_ranks(d: pd.DataFrame, rank_col: str):
    """Tính toán các chỉ số xếp hạng Recall@1, Recall@3, Recall@5 và điểm MRR."""
    # Recall@1: Tỷ lệ Bằng chứng Vàng nằm ngay vị trí số 1
    r1 = d[(d[rank_col] == 1) & (d['is_gold'] == True)]['claim_id'].nunique() / n_claims
    # Recall@3: Tỷ lệ Bằng chứng Vàng nằm trong Top 3
    r3 = d[(d[rank_col] <= 3) & (d['is_gold'] == True)]['claim_id'].nunique() / n_claims
    # Recall@5: Tỷ lệ Bằng chứng Vàng nằm trong Top 5
    r5 = d[(d[rank_col] <= 5) & (d['is_gold'] == True)]['claim_id'].nunique() / n_claims
    
    # Tính điểm trung bình nghịch đảo thứ hạng (MRR)
    g = d[d['is_gold'] == True]
    mrr = (1.0 / g.groupby('claim_id')[rank_col].min()).sum() / n_claims
    return r1, r3, r5, mrr

# Chấm điểm bảng xếp hạng cũ của BM25
bm25_r1, bm25_r3, bm25_r5, bm25_mrr = eval_ranks(eval_df, 'bm25_rank')
# Chấm điểm bảng xếp hạng mới sau khi kết hợp Fact Reranker
rerank_r1, rerank_r3, rerank_r5, rerank_mrr = eval_ranks(eval_df, 'rerank_rank')

# Đếm số lượng câu Bằng chứng Vàng ban đầu bị BM25 xếp ở Rank 2, 3, 4, 5
# nhưng nhờ Fact Reranker mà được 'thăng hạng' nhảy vọt lên Top 1
promoted = eval_df[(eval_df['bm25_rank'] > 1) & (eval_df['rerank_rank'] == 1) & (eval_df['is_gold'] == True)]

print('=' * 70)
print('SO SÁNH HIỆU NĂNG RETRIEVAL TRƯỚC VÀ SAU FACT RERANKING:')
print(f'• Recall@1: BM25 = {bm25_r1*100:.2f}%  -->  Fact Reranker = {rerank_r1*100:.2f}% (Chênh lệch: {(rerank_r1-bm25_r1)*100:+.2f}%)')
print(f'• MRR:      BM25 = {bm25_mrr:.4f}  -->  Fact Reranker = {rerank_mrr:.4f} (Chênh lệch: {rerank_mrr-bm25_mrr:+.4f})')
print(f'• Số câu Bằng chứng Vàng được thăng hạng lên Top-1: {len(promoted)} câu!')
print('=' * 70)


SO SÁNH HIỆU NĂNG RETRIEVAL TRƯỚC VÀ SAU FACT RERANKING:
• Recall@1: BM25 = 89.80%  -->  Fact Reranker = 90.00% (Delta: +0.20%)
• MRR:      BM25 = 0.9297  -->  Fact Reranker = 0.9299 (Delta: +0.0002)
• Số câu Bằng chứng Vàng được thăng hạng lên Top-1: 19 câu!


In [3]:
# ======================================================================
# TẠO CÁC TẬP DỮ LIỆU ĐẦU VÀO ĐỂ ĐƯA VÀO MÔ HÌNH NLI (BAMIBERT)
# ======================================================================
dev_cleaned = pd.read_csv(PROJECT_ROOT / 'data/processed/common_cleaned/vifactcheck_dev_common_cleaned.csv')

# 1. Dataset A: Gold Evidence (Bằng chứng Vàng chuẩn gán tay - Đối chứng lý tưởng)
gold_nli = pd.DataFrame({
    'claim_id': [f'dev_{i}' for i in dev_cleaned['index']],
    'claim_index': dev_cleaned['index'],
    'statement': dev_cleaned['Statement'],
    'evidence': dev_cleaned['Evidence'].fillna(''),
    'label': dev_cleaned['labels']
})

# 2. Dataset B: BM25 Top 1 Evidence (Lấy câu số 1 do BM25 thuần túy tìm thấy)
bm25_map = df_feats[df_feats['bm25_rank'] == 1].set_index('claim_index')['retrieved_evidence'].to_dict()
bm25_nli = pd.DataFrame({
    'claim_id': [f'dev_{i}' for i in dev_cleaned['index']],
    'claim_index': dev_cleaned['index'],
    'statement': dev_cleaned['Statement'],
    'evidence': dev_cleaned['index'].map(bm25_map).fillna(''),
    'label': dev_cleaned['labels']
})
bm25_nli.to_csv(OUTPUT_DIR / 'nli_input_bm25.csv', index=False)

# 3. Dataset C: Fact Reranked Top 1 Evidence (Lấy câu số 1 do Fact-aware Reranker chọn lọc)
rerank_map = df_feats[df_feats['rerank_rank'] == 1].set_index('claim_index')['retrieved_evidence'].to_dict()
rerank_nli = pd.DataFrame({
    'claim_id': [f'dev_{i}' for i in dev_cleaned['index']],
    'claim_index': dev_cleaned['index'],
    'statement': dev_cleaned['Statement'],
    'evidence': dev_cleaned['index'].map(rerank_map).fillna(''),
    'label': dev_cleaned['labels']
})
rerank_nli.to_csv(OUTPUT_DIR / 'nli_input_reranked.csv', index=False)

print(f'✓ Đã xuất các file NLI Input tại {OUTPUT_DIR}:')
print('  - nli_input_bm25.csv (Đầu vào từ BM25 thuần túy)')
print('  - nli_input_reranked.csv (Đầu vào sau khi lọc bằng Fact Reranker)')


✓ Đã xuất 3 file NLI Input tại /Users/mivu/Documents/Artificial Intelligence/CS221-NLP/Project-NLP-Fact-Checking/data/processed/retrieval:
  - nli_input_gold.csv
  - nli_input_bm25.csv
  - nli_input_reranked.csv


In [4]:
# ======================================================================
# ĐÁNH GIÁ NLI THỰC TẾ TRÊN MÔ HÌNH BAMIBERT SOTA (EXP006)
# ======================================================================
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Đường dẫn tới checkpoint mô hình tốt nhất BamiBERT Exp006
BAMIBERT_CKPT = PROJECT_ROOT / 'notebooks/models/bamibert/outputs/bamibert/experiments/Exp006_PrefixPrompt_SOTA/best_model'

print(f'Đang nạp mô hình BamiBERT từ: {BAMIBERT_CKPT.name}...')
tokenizer = AutoTokenizer.from_pretrained(BAMIBERT_CKPT)
model = AutoModelForSequenceClassification.from_pretrained(BAMIBERT_CKPT)

# Tự động chọn thiết bị tính toán tối ưu: Apple Silicon GPU (MPS) hoặc CUDA GPU hoặc CPU
device = torch.device('mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu'))
model.to(device)
model.eval()  # Chuyển mô hình sang chế độ suy luận (Inference mode)

def evaluate_nli_bamibert(df_nli: pd.DataFrame, batch_size: int = 32):
    """Đưa cặp Tuyên bố + Bằng chứng vào BamiBERT và đo Accuracy, Macro F1."""
    # Thêm tiền tố định danh vai trò theo kỹ thuật Prefix Prompting SOTA
    claims = [f'Tuyên bố: {str(s).strip()}' for s in df_nli['statement']]
    evidences = [f'Bằng chứng: {str(e).strip()}' for e in df_nli['evidence']]
    y_true = df_nli['label'].values  # Nhãn thật
    y_pred = []  # Danh sách nhãn do BamiBERT dự đoán
    
    with torch.no_grad():  # Tắt tính toán gradient để tăng tốc và tiết kiệm RAM
        for i in range(0, len(claims), batch_size):
            inputs = tokenizer(
                claims[i:i + batch_size],
                evidences[i:i + batch_size],
                max_length=256,
                padding=True,
                truncation=True,
                return_tensors='pt'
            ).to(device)
            logits = model(**inputs).logits
            # Chọn nhãn có xác suất cao nhất (argmax)
            y_pred.extend(torch.argmax(logits, dim=-1).cpu().numpy())
            
    acc = accuracy_score(y_true, y_pred)  # Tính độ chính xác
    macro_f1 = f1_score(y_true, y_pred, average='macro')  # Tính điểm cân bằng Macro F1
    return acc, macro_f1

# ======================================================================
# CHẠY ĐỐI CHỨNG TRỰC TIẾP TRÊN 3 NGUỒN BẰNG CHỨNG
# ======================================================================
print('Đang đánh giá Thí nghiệm A (Gold Evidence)...')
acc_a, f1_a = evaluate_nli_bamibert(gold_nli)
print('Đang đánh giá Thí nghiệm B (BM25 Top-1)...')
acc_b, f1_b = evaluate_nli_bamibert(bm25_nli)
print('Đang đánh giá Thí nghiệm C (Fact Reranked Top-1)...')
acc_c, f1_c = evaluate_nli_bamibert(rerank_nli)

# Bảng tổng hợp đối chứng hiệu năng
exp_summary = pd.DataFrame([
    {'Thí nghiệm (Experiment)': 'A. Gold Evidence (Upper bound)', 'Nguồn Bằng chứng': 'Bằng chứng chuẩn do con người gán', 'Accuracy': f'{acc_a*100:.2f}%', 'Macro-F1': f'{f1_a:.4f}'},
    {'Thí nghiệm (Experiment)': 'B. BM25 Evidence', 'Nguồn Bằng chứng': 'Bằng chứng Top 1 từ BM25 thuần túy', 'Accuracy': f'{acc_b*100:.2f}%', 'Macro-F1': f'{f1_b:.4f}'},
    {'Thí nghiệm (Experiment)': 'C. Fact-aware Reranked Evidence', 'Nguồn Bằng chứng': 'Bằng chứng Top 1 sau khi xếp hạng lại bằng Fact', 'Accuracy': f'{acc_c*100:.2f}%', 'Macro-F1': f'{f1_c:.4f}'}
])

display(exp_summary)
exp_summary.to_csv(OUTPUT_DIR / 'nli_comparison_metrics.csv', index=False)
print(f'✓ Đã lưu bảng chỉ số so sánh NLI tại: {OUTPUT_DIR / "nli_comparison_metrics.csv"}')

# ======================================================================
# VẼ BIỂU ĐỒ TRỰC QUAN HÓA SO SÁNH HIỆU NĂNG NLI TRÊN 3 NGUỒN BẰNG CHỨNG
# ======================================================================
labels = ['A. Bằng chứng Vàng\n(Gold Evidence)', 'B. BM25 Top-1\n(Từ vựng)', 'C. Fact Rerank Top-1\n(Xếp hạng Fact)']
accuracies = [acc_a * 100, acc_b * 100, acc_c * 100]
macro_f1s = [f1_a * 100, f1_b * 100, f1_c * 100]

x = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(9.5, 5.5))
rects1 = ax.bar(x - width/2, accuracies, width, label='Độ chính xác (Accuracy)', color='#3498db', edgecolor='black', linewidth=0.8)
rects2 = ax.bar(x + width/2, macro_f1s, width, label='Macro F1 (%)', color='#2ecc71', edgecolor='black', linewidth=0.8)

ax.set_title('Đối Sánh Hiệu Năng BamiBERT Trên 3 Nguồn Bằng Chứng (Tập Dev)', fontsize=13, weight='bold', pad=15)
ax.set_ylabel('Tỷ lệ (%)', fontsize=11, weight='bold')
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=10.5, weight='bold')
ax.set_ylim(0, 100)
ax.grid(axis='y', linestyle='--', alpha=0.5)

# Gắn nhãn % trực tiếp lên đầu từng cột bar
for rect in rects1:
    y = rect.get_height()
    ax.text(rect.get_x() + rect.get_width()/2, y + 1.2, f'{y:.2f}%', ha='center', va='bottom', fontsize=9.5, weight='bold')

for rect in rects2:
    y = rect.get_height()
    ax.text(rect.get_x() + rect.get_width()/2, y + 1.2, f'{y:.2f}%', ha='center', va='bottom', fontsize=9.5, weight='bold')

ax.legend(fontsize=10, loc='upper right', framealpha=0.95)
plt.tight_layout()
fig.savefig(OUTPUT_DIR / '04_nli_comparison_chart.png', dpi=200)
plt.show()
print(f'✓ Đã xuất biểu đồ so sánh NLI tại: {OUTPUT_DIR / "04_nli_comparison_chart.png"}')


Đang nạp mô hình BamiBERT từ: best_model...


,Thí nghiệm (Experiment),Loại Bằng chứng,Accuracy,Macro-F1
0,A. Gold Evidence (Upper bound),Bằng chứng chuẩn gán nhãn tay,81.05%,0.8116
1,B. BM25 Evidence,Bằng chứng Top 1 từ BM25 thuần túy,62.52%,0.6255
2,C. Fact-aware Reranked Evidence,Bằng chứng Top 1 sau khi Rerank đặc trưng Fact,62.38%,0.6242


✓ Đã lưu bảng chỉ số so sánh NLI tại: /Users/mivu/Documents/Artificial Intelligence/CS221-NLP/Project-NLP-Fact-Checking/data/processed/retrieval/nli_comparison_metrics.csv
